# Exploratory Data Analysis: Shakespeare Text Generation
This notebook outlines the character frequency distributions, text preprocessing verification, and data sequence lengths before running the production training pipeline.

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

# Append parent directory to path so we can import from 'src' and use 'config' smoothly
sys.path.append(os.path.abspath('..'))

from config import config
from src.data_loader import download_dataset
from src.preprocessor import TextPreprocessor

## 1. Load the Dataset
We load or download the text via our production data loader utility to check its size and characteristics.

In [ ]:
# Fetch dataset using default URL specified in central config
raw_text = download_dataset()
print(f"Total Characters in Raw Text: {len(raw_text):,}")
print("\n--- First 300 Characters Preview ---")
print(raw_text[:300])

## 2. Text Preprocessing & Cleaning Verification
We check the distribution changes after our regex cleaning pass (e.g., lowercasing, space collapsing, punctuation stripping).

In [ ]:
preprocessor = TextPreprocessor()
cleaned_text = preprocessor.clean_text(raw_text)
preprocessor.build_vocabulary(cleaned_text)

print(f"Cleaned Text Length: {len(cleaned_text):,} characters")
print(f"Reduction Ratio: {(1 - len(cleaned_text)/len(raw_text))*100:.2f}% of text removed as noise/formatting.")

## 3. Character Frequency Analysis
Let's see what characters are most common in our corpus to understand vocabulary weight imbalance.

In [ ]:
# Extract and count character occurrences
char_counts = Counter(cleaned_text)
most_common = char_counts.most_common(20)

chars, frequencies = zip(*most_common)

# Plotting distribution
plt.figure(figsize=(12, 5))
plt.bar(chars, frequencies, color='teal', edgecolor='black', alpha=0.7)
plt.title('Top 20 Most Frequent Characters in Preprocessed Corpus', fontsize=14, fontweight='bold')
plt.xlabel('Character Token', fontsize=12)
plt.ylabel('Total Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 4. Sliding Window & Input Tensor Structure
Lastly, we verify that our sliding window creates the correct tensor shapes ($X$ and $y$) required for training our stacked LSTM network.

In [ ]:
X, y = preprocessor.create_sequences(
    sequence_length=config.sequence_length,
    step_size=config.step_size
)

print("\n--- Tensor Shape Overview ---")
print(f"Input Features (X) Shape: {X.shape} -> (Number of sequences, Sequence length)")
print(f"Target Labels (y) Shape:  {y.shape} -> (Number of sequences,)")

print("\n--- Concrete Instance Preview ---")
sample_idx = 0
print(f"Sample Input Indices Vector: \n{X[sample_idx]}")
print(f"Decoded Input Text String:   \n'{preprocessor.decode_indices(X[sample_idx])}'")
print(f"Target Character Index:      {y[sample_idx]} -> Decoded Target: '{preprocessor.idx_to_char[y[sample_idx]]}'")